# FraudShield Advanced: Inference Patterns via SDK

## Concept Review: Inference Decision Matrix

SageMaker provides four inference deployment options. Choosing the wrong one
leads to wasted cost or unacceptable latency. Evaluate every deployment against
five criteria:

1. **Latency requirement** -- sub-second, seconds, or minutes?
2. **Traffic pattern** -- steady, bursty, sporadic, or one-time?
3. **Payload size** -- kilobytes, megabytes, or gigabytes?
4. **Processing time** -- milliseconds, seconds, or minutes?
5. **Cost sensitivity** -- minimize cost or maximize performance?

| Criteria | Real-time Endpoint | Serverless | Async Inference | Batch Transform |
|---|---|---|---|---|
| **Latency** | < 1 second | 1-30 seconds (cold start) | Minutes | Hours (job-based) |
| **Traffic** | Steady or predictable | Sporadic, low volume | Variable, queued | One-time or scheduled |
| **Max payload** | 6 MB | 6 MB | 1 GB | Unlimited |
| **Max processing** | 60 seconds | 60 seconds | 60 minutes | No limit |
| **Scale to zero** | No | Yes | Yes | N/A |
| **GPU support** | Yes | No | Yes | Yes |
| **Billing model** | Instance-hours | Compute ms + data | Instance-hours (0 when idle) | Instance-hours (job duration) |

Decision flowchart:

1. Bulk scoring with no latency requirement? --> Batch Transform
2. Payload > 6 MB or processing > 60 seconds? --> Async Inference
3. Sub-second response required? --> Real-time Endpoint
4. Sporadic, low-volume traffic? --> Serverless Inference

This notebook demonstrates Serverless, Async, Batch Transform, Multi-Model, and
Multi-Container patterns using the SageMaker Python SDK. It is organized in
three stages:

1. **Serverless and Async Inference** -- scale-to-zero patterns for variable traffic
2. **Batch Transform** -- ephemeral compute for bulk offline scoring
3. **Multi-Model and Multi-Container Endpoints** -- shared infrastructure for multiple models

---

*This notebook runs on ml.t3.medium in SageMaker Studio. Inference endpoints
use ml.m5.xlarge (Free Tier eligible).*

In [ ]:
%pip install -q sagemaker boto3 pandas

In [ ]:
import sagemaker
import boto3
import json
import time
from datetime import datetime
from sagemaker.model import Model
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.async_inference.async_inference_config import AsyncInferenceConfig
from sagemaker.multidatamodel import MultiDataModel
from sagemaker.pipeline import PipelineModel
from sagemaker.transformer import Transformer
from sagemaker import image_uris

In [ ]:
sm_session = sagemaker.Session()
sm_client = sm_session.sagemaker_client
sm_runtime = boto3.client("sagemaker-runtime")
s3_client = boto3.client("s3")
region = sm_session.boto_region_name
role = sagemaker.get_execution_role()
default_bucket = sm_session.default_bucket()

PREFIX = "fraudshield-inference"
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

MODEL_ARTIFACT = f"s3://{default_bucket}/{PREFIX}/models/model.tar.gz"
CONTAINER_IMAGE = image_uris.retrieve("xgboost", region, version="1.5-1")

test_payload = "0.5,1.0,0.3,0.7,0.2,0.9,0.1,0.4,0.6,0.8"

print(f"Region          : {region}")
print(f"Role            : {role.split('/')[-1]}")
print(f"Default bucket  : {default_bucket}")
print(f"Container image : {CONTAINER_IMAGE}")
print(f"Model artifact  : {MODEL_ARTIFACT}")
print()
print("NOTE: This notebook assumes a trained XGBoost model artifact exists")
print("at the MODEL_ARTIFACT path above. If you have not trained a model yet,")
print("update MODEL_ARTIFACT to point to your model.tar.gz in S3.")

---

## Stage 1: Serverless and Asynchronous Inference

### Concept Review: Serverless Inference

Serverless Inference eliminates idle costs by scaling compute to zero when there
are no requests. You configure two parameters:

- **MemorySizeInMB:** RAM allocated to each inference container (1024-6144 MB,
  in 1024 MB increments). Larger memory supports bigger models and faster
  deserialization.
- **MaxConcurrency:** Maximum concurrent requests the endpoint handles (1-200).
  SageMaker provisions additional containers up to this limit.

The trade-off is **cold start latency**. When the first request arrives after
idle time, SageMaker must:

1. Provision a compute instance
2. Download the model artifact from S3
3. Load the model into memory
4. Run inference

Cold start typically adds 5-30 seconds depending on model size. Subsequent
requests reuse the warm container and respond in milliseconds.

**Cost comparison:** A Serverless endpoint handling 1,000 requests/day at 200ms
each costs roughly $3-5/month vs. ~$170/month for an always-on ml.m5.xlarge.

**Limitations:** 6 MB max payload, 60 second max response time, CPU only, no
multi-model endpoints.

**Mitigation strategies for cold start:**
- Provisioned Concurrency: keep minimum containers warm (adds base cost)
- Model size optimization: smaller artifacts download faster
- Warm-up requests: periodic dummy invocations during expected traffic hours

In [ ]:
serverless_model_name = f"fraudshield-serverless-{TIMESTAMP}"

serverless_model = Model(
    model_data=MODEL_ARTIFACT,
    image_uri=CONTAINER_IMAGE,
    role=role,
    sagemaker_session=sm_session,
    name=serverless_model_name,
)

print(f"Model object created: {serverless_model_name}")
print(f"  Image: {CONTAINER_IMAGE}")
print(f"  Data:  {MODEL_ARTIFACT}")

In [ ]:
serverless_endpoint_name = f"fraudshield-serverless-ep-{TIMESTAMP}"

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5,
)

serverless_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name=serverless_endpoint_name,
)

print(f"Serverless endpoint deployed: {serverless_endpoint_name}")
print(f"  Memory: 2048 MB")
print(f"  Max concurrency: 5")

In [ ]:
print("Invoking serverless endpoint (first call -- expect cold start)...")
start = time.time()
response = sm_runtime.invoke_endpoint(
    EndpointName=serverless_endpoint_name,
    ContentType="text/csv",
    Body=test_payload,
)
cold_latency = time.time() - start
result = response["Body"].read().decode("utf-8")

print(f"Response          : {result}")
print(f"Cold start latency: {cold_latency:.2f} seconds")
print()
print("The cold start includes instance provisioning, model download")
print("from S3, and model deserialization into memory.")

In [ ]:
print("Invoking serverless endpoint (second call -- container is warm)...")
start = time.time()
response = sm_runtime.invoke_endpoint(
    EndpointName=serverless_endpoint_name,
    ContentType="text/csv",
    Body=test_payload,
)
warm_latency = time.time() - start
result = response["Body"].read().decode("utf-8")

print(f"Response       : {result}")
print(f"Warm latency   : {warm_latency:.2f} seconds")
print()
print(f"Cold vs Warm comparison:")
print(f"  Cold start   : {cold_latency:.2f}s")
print(f"  Warm request : {warm_latency:.2f}s")
print(f"  Speedup      : {cold_latency / max(warm_latency, 0.001):.1f}x")
print()
print("The warm call reuses the already-provisioned container.")
print("For FraudShield, this cold start is acceptable for internal")
print("compliance tools but not for real-time card swipe scoring.")

### Concept Review: Asynchronous Inference

Async Inference decouples the request from the response. The workflow:

1. Client uploads input to S3
2. Client invokes the endpoint with `InvokeEndpointAsync`, passing the S3 URI
3. SageMaker queues the request and returns immediately with an output location
4. SageMaker processes asynchronously, writing results to S3
5. Optionally, SNS notifications signal completion

Key advantages over Serverless:

| Factor | Serverless | Async |
|---|---|---|
| Max payload | 6 MB | 1 GB |
| Max processing time | 60 seconds | 60 minutes |
| GPU support | No | Yes |
| Scale to zero | Yes | Yes |
| Latency | Seconds (cold start) | Minutes |

Async is ideal for large-payload, long-running predictions where the client
can tolerate minutes of latency -- document processing, image analysis, or
complex ensemble scoring. SNS notifications enable event-driven downstream
processing (Lambda, SQS) without polling.

In [ ]:
print("Deleting serverless endpoint before redeploying as async...")
sm_client.delete_endpoint(EndpointName=serverless_endpoint_name)
print(f"Deleted endpoint: {serverless_endpoint_name}")

time.sleep(10)

async_endpoint_name = f"fraudshield-async-ep-{TIMESTAMP}"
async_model_name = f"fraudshield-async-{TIMESTAMP}"
s3_async_output = f"s3://{default_bucket}/{PREFIX}/async-output/"

async_model = Model(
    model_data=MODEL_ARTIFACT,
    image_uri=CONTAINER_IMAGE,
    role=role,
    sagemaker_session=sm_session,
    name=async_model_name,
)

async_config = AsyncInferenceConfig(
    output_path=s3_async_output,
)

async_model.deploy(
    instance_type="ml.m5.xlarge",
    initial_instance_count=1,
    async_inference_config=async_config,
    endpoint_name=async_endpoint_name,
)

print(f"Async endpoint deployed: {async_endpoint_name}")
print(f"  Instance type: ml.m5.xlarge")
print(f"  Output path  : {s3_async_output}")

In [ ]:
input_key = f"{PREFIX}/async-input/test-request.csv"
s3_client.put_object(
    Bucket=default_bucket,
    Key=input_key,
    Body=test_payload.encode("utf-8"),
)
input_s3_uri = f"s3://{default_bucket}/{input_key}"
print(f"Uploaded test input to: {input_s3_uri}")

response = sm_runtime.invoke_endpoint_async(
    EndpointName=async_endpoint_name,
    InputLocation=input_s3_uri,
    ContentType="text/csv",
)
output_location = response["OutputLocation"]
print(f"Async invocation accepted.")
print(f"Output will be at: {output_location}")

print("\nPolling S3 for result...")
output_bucket = output_location.split("/")[2]
output_key = "/".join(output_location.split("/")[3:])

for attempt in range(30):
    try:
        result_obj = s3_client.get_object(Bucket=output_bucket, Key=output_key)
        result_body = result_obj["Body"].read().decode("utf-8")
        print(f"Result received after ~{(attempt + 1) * 10} seconds:")
        print(f"  Prediction: {result_body}")
        break
    except s3_client.exceptions.NoSuchKey:
        time.sleep(10)
else:
    print("Timed out waiting for async result. Check S3 output path manually.")

In [ ]:
print("Deleting async endpoint...")
sm_client.delete_endpoint(EndpointName=async_endpoint_name)
print(f"Deleted endpoint: {async_endpoint_name}")

---

## Stage 2: Batch Transform

### Concept Review: Batch Transform Architecture

Batch Transform is the simplest inference option: no endpoint to manage, no
scaling to configure, no cleanup to remember. The lifecycle:

1. You create a transform job specifying model, S3 input, S3 output, instance
   type, and instance count
2. SageMaker provisions an ephemeral cluster and loads the model on each instance
3. Input data is split across instances (ShardedByS3Key by default)
4. Predictions are written to S3 -- output files mirror input names with `.out`
5. SageMaker tears down the cluster when complete

Key configuration options:

- **Split type:** `Line` (each line is a record) or `None` (each file is one record)
- **Join source:** Set to `Input` to append predictions alongside original records,
  essential for matching predictions back to customer IDs or transaction IDs
- **Data distribution:** `ShardedByS3Key` (parallel across instances) or
  `FullyReplicated` (every instance gets all data)
- **Max payload size:** Maximum MB per request sent to the model container
- **Max concurrent transforms:** Records processed in parallel per instance

Batch Transform is ideal for bulk scoring: nightly re-ranking, retroactive
analysis, generating labels from a teacher model. You pay only for the job
duration -- no persistent endpoint to clean up.

In [ ]:
batch_data_key = f"{PREFIX}/batch-input/transactions_batch.csv"

batch_rows = []
for i in range(50):
    row = ",".join([f"{(i * 0.02 + j * 0.1) % 1:.2f}" for j in range(10)])
    batch_rows.append(row)
batch_body = "\n".join(batch_rows)

s3_client.put_object(
    Bucket=default_bucket,
    Key=batch_data_key,
    Body=batch_body.encode("utf-8"),
)

batch_input_s3 = f"s3://{default_bucket}/{batch_data_key}"
batch_output_s3 = f"s3://{default_bucket}/{PREFIX}/batch-output/"

print(f"Uploaded {len(batch_rows)} synthetic transaction records")
print(f"Input : {batch_input_s3}")
print(f"Output: {batch_output_s3}")

In [ ]:
batch_model_name = f"fraudshield-batch-{TIMESTAMP}"

batch_model = Model(
    model_data=MODEL_ARTIFACT,
    image_uri=CONTAINER_IMAGE,
    role=role,
    sagemaker_session=sm_session,
    name=batch_model_name,
)
batch_model.create(instance_type="ml.m5.xlarge")

print(f"Model registered for batch transform: {batch_model_name}")

In [ ]:
transformer = Transformer(
    model_name=batch_model_name,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=batch_output_s3,
    sagemaker_session=sm_session,
    accept="text/csv",
    strategy="SingleRecord",
)

print(f"Transformer object created")
print(f"  Model        : {batch_model_name}")
print(f"  Instance type: ml.m5.xlarge")
print(f"  Instance count: 1")
print(f"  Strategy     : SingleRecord")
print(f"  Output path  : {batch_output_s3}")

In [ ]:
print("Starting batch transform job...")
transform_start = time.time()

transformer.transform(
    data=batch_input_s3,
    content_type="text/csv",
    split_type="Line",
)

print("Transform job submitted. Waiting for completion...")
transformer.wait()

transform_duration = time.time() - transform_start
print(f"Batch transform complete in {transform_duration:.0f} seconds.")

In [ ]:
output_key = f"{PREFIX}/batch-output/transactions_batch.csv.out"

try:
    result_obj = s3_client.get_object(Bucket=default_bucket, Key=output_key)
    result_body = result_obj["Body"].read().decode("utf-8")
    predictions = result_body.strip().split("\n")

    print(f"Total predictions returned: {len(predictions)}")
    print(f"Expected records:           {len(batch_rows)}")
    print(f"\nFirst 10 predictions:")
    for i, pred in enumerate(predictions[:10]):
        print(f"  Record {i:3d}: {pred}")
except Exception as e:
    print(f"Error reading output: {e}")
    print(f"Check S3 path: s3://{default_bucket}/{output_key}")

In [ ]:
print("Batch Transform output format:")
print(f"  Input file  : transactions_batch.csv")
print(f"  Output file : transactions_batch.csv.out")
print(f"  Each line in the output corresponds to the same line in the input.")
print(f"  With 'join_source=Input', SageMaker appends predictions to input rows.")
print()
print("No endpoint was created -- the cluster was ephemeral.")
print("There is no endpoint to delete or scaling to manage.")

In [ ]:
print(f"Cleaning up batch model resource: {batch_model_name}")
sm_client.delete_model(ModelName=batch_model_name)
print("Batch model deleted.")

---

## Stage 3: Multi-Model Endpoints

### Concept Review: Multi-Model Endpoints (MME)

Multi-Model Endpoints host multiple models on shared infrastructure. Instead of
deploying one endpoint per model (at ~$170/month each), you deploy a single
endpoint that dynamically loads models on demand.

How it works:

1. Model artifacts (.tar.gz) are stored under a common S3 prefix
2. A single endpoint points to this S3 prefix
3. The client specifies which model to invoke via the `TargetModel` parameter
4. SageMaker checks if the model is in memory:
   - **Loaded (hot):** Routes request directly, same latency as single-model
   - **Not loaded (cold):** Downloads from S3, loads into memory, then infers
5. When memory is full, the **Least Recently Used (LRU)** model is evicted

Constraints:
- All models must use the same inference container (same framework and version)
- All models share the same instance type
- No guaranteed memory reservation per model

Adding a new model requires only uploading a .tar.gz to the S3 prefix -- no
endpoint redeployment needed. Removing a model means deleting it from S3.

MME can reduce costs by 90%+ for multi-model serving scenarios (e.g.,
per-region fraud models, per-customer personalization models).

In [ ]:
mme_s3_prefix = f"s3://{default_bucket}/{PREFIX}/mme-models/"

model_names = ["model_a.tar.gz", "model_b.tar.gz", "model_c.tar.gz"]

source_bucket = MODEL_ARTIFACT.split("/")[2]
source_key = "/".join(MODEL_ARTIFACT.split("/")[3:])

for model_name in model_names:
    dest_key = f"{PREFIX}/mme-models/{model_name}"
    s3_client.copy_object(
        CopySource={"Bucket": source_bucket, "Key": source_key},
        Bucket=default_bucket,
        Key=dest_key,
    )
    print(f"Uploaded: s3://{default_bucket}/{dest_key}")

print(f"\n3 model artifacts staged at prefix: {mme_s3_prefix}")

In [ ]:
mme_model_name = f"fraudshield-mme-{TIMESTAMP}"

mme_model = MultiDataModel(
    name=mme_model_name,
    model_data_prefix=mme_s3_prefix,
    image_uri=CONTAINER_IMAGE,
    role=role,
    sagemaker_session=sm_session,
)

print(f"MultiDataModel created: {mme_model_name}")
print(f"  S3 prefix: {mme_s3_prefix}")
print(f"  Models   : {model_names}")

In [ ]:
mme_endpoint_name = f"fraudshield-mme-ep-{TIMESTAMP}"

mme_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=mme_endpoint_name,
)

print(f"Multi-Model Endpoint deployed: {mme_endpoint_name}")
print(f"  Instance type : ml.m5.xlarge")
print(f"  Instance count: 1")

In [ ]:
print("Invoking with TargetModel='model_a.tar.gz'...")
start = time.time()
response_a = sm_runtime.invoke_endpoint(
    EndpointName=mme_endpoint_name,
    TargetModel="model_a.tar.gz",
    ContentType="text/csv",
    Body=test_payload,
)
latency_a = time.time() - start
result_a = response_a["Body"].read().decode("utf-8")
print(f"  Result : {result_a}")
print(f"  Latency: {latency_a:.2f}s (first load -- model downloaded from S3)")

In [ ]:
print("Invoking with TargetModel='model_b.tar.gz'...")
start = time.time()
response_b = sm_runtime.invoke_endpoint(
    EndpointName=mme_endpoint_name,
    TargetModel="model_b.tar.gz",
    ContentType="text/csv",
    Body=test_payload,
)
latency_b = time.time() - start
result_b = response_b["Body"].read().decode("utf-8")
print(f"  Result : {result_b}")
print(f"  Latency: {latency_b:.2f}s (first load for model_b)")

In [ ]:
print("Invoking with TargetModel='model_c.tar.gz'...")
start = time.time()
response_c = sm_runtime.invoke_endpoint(
    EndpointName=mme_endpoint_name,
    TargetModel="model_c.tar.gz",
    ContentType="text/csv",
    Body=test_payload,
)
latency_c = time.time() - start
result_c = response_c["Body"].read().decode("utf-8")
print(f"  Result : {result_c}")
print(f"  Latency: {latency_c:.2f}s (first load for model_c)")

print("\nRe-invoking model_a (should be cached -- LRU hot):")
start = time.time()
response_a2 = sm_runtime.invoke_endpoint(
    EndpointName=mme_endpoint_name,
    TargetModel="model_a.tar.gz",
    ContentType="text/csv",
    Body=test_payload,
)
latency_a_cached = time.time() - start
result_a2 = response_a2["Body"].read().decode("utf-8")
print(f"  Result : {result_a2}")
print(f"  Latency: {latency_a_cached:.2f}s (cached -- no S3 download)")

print(f"\nLatency comparison:")
print(f"  model_a (cold): {latency_a:.2f}s")
print(f"  model_b (cold): {latency_b:.2f}s")
print(f"  model_c (cold): {latency_c:.2f}s")
print(f"  model_a (hot) : {latency_a_cached:.2f}s")

### Concept Review: Multi-Container Endpoints

Multi-Container Endpoints solve a different problem from MME: running a
*pipeline* of different model containers on a single endpoint.

**Two invocation modes:**

- **Serial inference pipeline:** Containers are chained in sequence. The output
  of Container A becomes the input of Container B. The client sends one request
  and the final container's response is returned.
- **Direct invocation:** The client specifies which container to call using
  `TargetContainerHostname`. Useful for hosting unrelated models that use
  different frameworks on shared infrastructure.

A typical serial pipeline:

1. **Container A (Preprocessing):** Transforms raw input into model-ready features
2. **Container B (Inference):** Runs the ML model, returns predictions
3. **Container C (Post-processing, optional):** Applies business rules

Advantages over monolithic containers:

| Factor | Multi-Container Pipeline | Monolithic Container |
|---|---|---|
| Independence | Each container versioned independently | Single image, full rebuild on any change |
| Reusability | Share preprocessor across endpoints | Logic duplicated per model |
| Framework mixing | Each container can use different framework | All code in one environment |
| Debugging | Isolate issues to specific container | All logic in one container |

Limitations: Max 15 containers, shared instance resources, strictly linear
pipelines (no branching or conditional routing).

In [ ]:
pipeline_model_name = f"fraudshield-pipeline-{TIMESTAMP}"

preprocessor_container = Model(
    model_data=MODEL_ARTIFACT,
    image_uri=CONTAINER_IMAGE,
    role=role,
    sagemaker_session=sm_session,
    env={"CONTAINER_ROLE": "preprocessor"},
)

predictor_container = Model(
    model_data=MODEL_ARTIFACT,
    image_uri=CONTAINER_IMAGE,
    role=role,
    sagemaker_session=sm_session,
    env={"CONTAINER_ROLE": "predictor"},
)

pipeline_model = PipelineModel(
    name=pipeline_model_name,
    role=role,
    models=[preprocessor_container, predictor_container],
    sagemaker_session=sm_session,
)

print(f"PipelineModel created: {pipeline_model_name}")
print(f"  Container 1: preprocessor")
print(f"  Container 2: predictor")
print()
print("In production, the preprocessor would use a custom container image")
print("with feature engineering logic. Here both containers use the XGBoost")
print("image to demonstrate the PipelineModel API pattern without requiring")
print("a custom Docker build.")

In [ ]:
print("PipelineModel deployment summary:")
print(f"  Model name  : {pipeline_model_name}")
print(f"  Containers  : 2 (preprocessor --> predictor)")
print(f"  Data flow   : Client request --> Container 1 output --> Container 2 --> Response")
print()
print("To deploy this PipelineModel as a live endpoint, call:")
print("  pipeline_model.deploy(")
print("      initial_instance_count=1,")
print("      instance_type='ml.m5.xlarge',")
print("  )")
print()
print("We skip the actual deployment here to avoid creating an additional")
print("billable endpoint. The pattern is identical to single-model deploy().")

---

## Mandatory Cleanup

All endpoints, endpoint configurations, and models created in this notebook
must be deleted to avoid ongoing charges. The cells below delete every resource
created during this session and verify none remain.

In [ ]:
endpoints_to_delete = [
    serverless_endpoint_name,
    async_endpoint_name,
    mme_endpoint_name,
]

models_to_delete = [
    serverless_model_name,
    async_model_name,
    batch_model_name,
    mme_model_name,
    pipeline_model_name,
]

print("=" * 60)
print("CLEANUP: Deleting all inference resources")
print("=" * 60)

for ep_name in endpoints_to_delete:
    try:
        sm_client.delete_endpoint(EndpointName=ep_name)
        print(f"  Deleted endpoint: {ep_name}")
    except sm_client.exceptions.ClientError:
        print(f"  Endpoint already deleted or not found: {ep_name}")

    try:
        sm_client.delete_endpoint_config(EndpointConfigName=ep_name)
        print(f"  Deleted endpoint config: {ep_name}")
    except sm_client.exceptions.ClientError:
        print(f"  Endpoint config already deleted or not found: {ep_name}")

for model_name in models_to_delete:
    try:
        sm_client.delete_model(ModelName=model_name)
        print(f"  Deleted model: {model_name}")
    except sm_client.exceptions.ClientError:
        print(f"  Model already deleted or not found: {model_name}")

In [ ]:
print("=" * 60)
print("VERIFICATION: Checking for remaining resources")
print("=" * 60)

remaining_eps = sm_client.list_endpoints(
    NameContains="fraudshield",
    StatusEquals="InService",
)["Endpoints"]

if remaining_eps:
    print(f"\n  WARNING: {len(remaining_eps)} endpoint(s) still active:")
    for ep in remaining_eps:
        print(f"    - {ep['EndpointName']} ({ep['EndpointStatus']})")
    print("  Delete these manually before closing this session.")
else:
    print("  No active FraudShield endpoints found.")

remaining_models = sm_client.list_models(
    NameContains="fraudshield",
    MaxResults=20,
)["Models"]

if remaining_models:
    print(f"\n  WARNING: {len(remaining_models)} model(s) still registered:")
    for m in remaining_models:
        print(f"    - {m['ModelName']}")
else:
    print("  No FraudShield models found.")

print("\nCleanup complete.")